# 1. Data and the five invariant fields

**Goal of the series.** Starting from raw robot tracks, these notebooks rebuild every step of the method of the paper
*Topological Kinematics of Collective Motion*, one step per notebook:

1. data and the five fields (this notebook);
2. scales, the grid-free domain and extended persistence;
3. distances between recordings and their fusion;
4. the taxonomy: affinity propagation, plateau, exemplars;
5. the hierarchy of missions and what each field explains;
6. ablation, baselines and robustness.

They read the results computed by `run_pipeline.py` and `run_sweep.py` (in `results/`), so they run in seconds.

**The data.** 20 e-puck robots in a dodecagonal arena of 4.91 m$^2$ (ARGoS simulator), positions every 0.1 s, controlled
by AutoMoDe-Chocolate finite-state machines from Hasselmann et al. (Nat. Commun. 2021), on five missions:
Directional Gate (120 s), Foraging (180 s), Homing (120 s), Shelter (180 s), XOR Aggregation (180 s).
Ten controllers per mission, 50 runs each. **Top-$n$** pools the $n$ best controllers of every mission ($250n$ runs).

In [ ]:
import sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "src"), str(ROOT)]
import numpy as np
import matplotlib.pyplot as plt
import figlib as L
L.style()
PAPER = ROOT / "results" / "epuck" / "paper"
RES = ROOT / "results" / "epuck"
load = lambda p: pickle.load(open(p, "rb"))

In [ ]:
d = load(PAPER / "top1_tracks.pkl")          # Top-1: one controller per mission, 250 runs
tracks, y = d["tracks"], d["labels"]
from collections import Counter
print("runs per mission:", {L.MISSIONS[m]: c for m, c in sorted(Counter(y.tolist()).items())})
print("frames per run (T):", sorted(set(t["pos"].shape[0] for t in tracks)), " robots:", tracks[0]["pos"].shape[1])

## A recording

A recording is the set of tracks $x_i(t)$ of the $N=20$ robots. Velocities are forward differences
$v_i(t) = (x_i(t+\Delta t)-x_i(t))/\Delta t$, i.e. the momenta of unit-mass agents. **Positions and velocities are all
the method sees** (no sensors, no environment, no labels).

In [ ]:
import make_figures as M
first = [int(np.where(y == m)[0][0]) for m in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(12, 2.8))
for ax, e in zip(axes, first):
    M.draw_tracks(ax, tracks[e]["pos"], L.MCOL[int(y[e])])
    ax.set_title(L.MSHORT[int(y[e])])
plt.show()

## From agents to fields: the Gaussian footprint and the three moments

Each robot is spread over a Gaussian footprint $O_i(r,t)=\exp(-\|r-x_i(t)\|^2/2\sigma^2)$ of width $\sigma$, the
observation scale. Weighted by the footprint, the velocities have three moments:

$$M_0=\sum_i O_i,\qquad M_1=\sum_i v_i O_i,\qquad M_2=\sum_i v_i\otimes v_i\,O_i .$$

None is redundant: a group at rest has $M_1=0$ but occupies space ($M_0$); two crossing streams cancel in $M_1$ but add
up in $\mathrm{tr}\,M_2$ (kinetic intensity).

**Rotation invariance.** A behaviour does not depend on the orientation of the scene, so we keep only quantities that
rotations leave unchanged:

* $\rho_\sigma$ = $M_0/N$ averaged over the recording (density);
* $K_\sigma$ = $\mathrm{tr}\,M_2/(N\overline{v^2})$ averaged (kinetic intensity, the spin-0 part of $M_2$);
* $M_1$ is a vector (spin 1): it is reduced by the **Helmholtz-Hodge decomposition**
  $V_\sigma=\nabla\varphi_\sigma+\nabla^\perp\psi_\sigma$ into a divergence potential $\varphi_\sigma$ (expansion /
  contraction) and a vorticity potential $\psi_\sigma$ (rotation), both in closed form;
* $J_\sigma=\sigma^2\det(\nabla\varphi_\sigma,\nabla\psi_\sigma)$ couples the two potentials (see notebook 2 for why it is needed).

The footprint is evaluated within $5\sigma$; the mass outside that window is $e^{-12.5}$:

In [ ]:
print(f"fraction of the footprint mass beyond 5 sigma: {np.exp(-12.5):.3e}")

## The five fields of one recording

The pipeline evaluates the fields exactly at grid-free seeds (notebook 2). For display we evaluate the same closed forms
on a regular grid. White = no data (farther than $5\sigma$ from every robot at every instant); zero is light blue for
$\rho, K$ and light grey for the signed fields.

In [ ]:
cache = load(PAPER / "display_fields.pkl")
ladder = load(RES / "scale_ladder.pkl")["ladder"]
tax = load(PAPER / "taxonomy.pkl")[1] if (PAPER / "taxonomy.pkl").exists() else None
e = int(tax["exemplars"][0]) if tax is not None else 175        # a Shelter exemplar
F = cache[("ladder", e, 2)]                                      # scale sigma_3
fig, axes = plt.subplots(1, 5, figsize=(13, 2.9))
for ax, f in zip(axes, L.FIELDS):
    L.show_field(ax, F[f], f); ax.set_title(M.FCOL_TITLE[f])
plt.show()

### Recomputing the fields yourself (CPU, about a minute)

The next cell evaluates the closed forms of `src/gaussian_splatting/fields_cpu.py` on a coarse 50 x 50 grid. It shows
that the displayed fields are not interpolated: they are the analytic sums over robots and instants.

In [ ]:
F50 = L.display_fields(tracks[e]["pos"], tracks[e]["vel"], float(ladder[2]), n=50, gpu=False, chunk=60)
fig, axes = plt.subplots(1, 5, figsize=(13, 2.9))
for ax, f in zip(axes, L.FIELDS):
    L.show_field(ax, F50[f], f); ax.set_title(f)
plt.show()

### Rotating the scene rotates the fields, and changes nothing else

Rotate the whole recording by 90 degrees about the arena centre: the fields of the rotated recording are the rotated
fields of the original (Proposition 2 of the paper). The distances between diagrams are therefore unchanged; on the
robot data this is checked numerically in notebook 6.

In [ ]:
c = L.ARENA / 2
R = np.array([[0, -1], [1, 0]])                                  # rotation by +90 degrees
pos_r = (tracks[e]["pos"] - c) @ R.T + c; vel_r = tracks[e]["vel"] @ R.T
F50r = L.display_fields(pos_r.astype(np.float32), vel_r.astype(np.float32), float(ladder[2]), n=50, gpu=False, chunk=60)
for f in L.FIELDS:
    a, b = np.rot90(F50[f], k=-1), F50r[f]                               # rotate the original field image by +90 degrees
    m = np.isfinite(a) & np.isfinite(b)
    print(f"{f:4s} max |field(rotated run) - rotated field| / max|field| = {np.abs(a[m]-b[m]).max()/np.abs(a[m]).max():.2e}")